# 3.5 Exercise: what did that capacity buy on the heartbeats?

Notebook 3.1 put a router next to a plain layer of the same parameter count and found
that, on the electricity series, the router bought nothing the parameters did not.
Notebook 3.2 drew what parameters, data and compute buy as curves with their spread.
Here both come together on lesson 2's heartbeats: you add routing to a design of a beat,
put its capacity-matched control next to it, and measure a capacity curve over two axes.
At the end you can say, from your own runs, on which axis this problem is bound and
whether routing earned its parameters, by more than the spread between runs.

## What is asked of you

1. **Part 1: run this notebook as it is.** The baseline, a capacity curve over
   parameters and one over data, each with the route stage and its control, and one
   handed-out search over every capacity axis. You change nothing; you read what comes
   out and answer the questions for yourself.
2. **Part 2: one study of your own**, in the folder [`exercises/`](exercises/): one idea,
   a design that can test it, a search over at least two capacity axes, and the route
   stage against its control. The class spreads over the open questions at the end of
   this notebook, and builds on each other's ideas.
3. **Hand in your runs** before the next lesson, with one command: `make share LESSON=3`.

## Where your runs go

As in lessons 1 and 2: every run is written to `artefacts/optuna/studies.db` on your
machine, `make optuna-dashboard` shows it, and `make share LESSON=3` hands in the runs of
lesson 3. Part 1 writes to two studies, `lesson3-notebook` (the trials of the search) and
`lesson3-notebook-runs` (the baseline and every point of the two curves). Part 2 writes to
`lesson3-study-<your study>` and its `-runs`.

**Nothing in Part 1 is seeded**, so your numbers will not be your neighbour's. The spread
between runs is what tells you how large a difference has to be before it means anything.

## The problem, and the three axes of capacity

The data is lesson 2's: single beats from the MIT-BIH Arrhythmia Database, three classes
(N, S, V), split by patient, scored on six patients of train that no model trains on
(notebook 2.4). The metric is again **macro recall**, the mean of the recalls of N, S and
V, so "always N" scores 1/3.

A run spends capacity on three axes, and the harness of this lesson lets you turn each:

| axis | what you turn | what a run logs |
|---|---|---|
| parameters | `width` (channels) and `depth` (blocks) of the design | `n_params` |
| data | `train_fraction`: the share of the training beats a run trains on | `n_train` |
| compute | `epochs`: the passes over those beats | `epochs` |

`train_fraction` is taken **stratified**: the beats of every class are shuffled once, with
a fixed seed, and a fraction keeps the first part of every class. A run on 0.2 trains on
the same mix of N, S and V as a run on 1.0, and on a subset of the beats of a run on 0.5,
so moving along the data axis changes the amount of data and nothing else.

## Setup

The data, the trainer and the design come from the folder `exercises/`, the same code
your own study uses in Part 2:

| name | what it is |
|---|---|
| `HARNESS.data(config)` | the train loader (the `train_fraction` of the training beats) and the validation loader |
| `HARNESS.fit(model, data, config)` | trains a model and scores it on the validation patients after every epoch |
| `DESIGN.build(config, n_classes=3)` | the design of [`exercises/designs.py`](exercises/designs.py): lesson 2's `Beats` with the options `route` and `matched` |
| `spread(config)` | `REPEATS` runs of one config, each from other random weights, logged to `RUNS` |
| `search(space, name)` | `N_TRIALS` trials of a search space as one Optuna study |
| `curve(table, axis)` | the trials of a search along one axis, per arm: mean, standard error, and the route arm against its control in sigmas |
| `RUNS` | the study `lesson3-notebook-runs`; `RUNS.log(...)` stores one run in it |

In [ ]:
import sys
import time

import datasets
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import torch
from exercises.designs import DESIGN, matched_hidden
from exercises.harness import (
    CHOICES,
    CLASSES,
    HARNESS,
    LINEAR,
    TRAIN_ROWS,
    arm,
    curve,
    fraction,
    splits,
    trial_rows,
)
from huggingface_hub.utils import disable_progress_bars as hub_quiet
from loguru import logger
from notebooktester import param
from torch import nn

from hpo.measure import Architecture, Comparison, Confusion, repeat
from hpo.record import Idea, RunRecord, StudyLog
from hpo.search import Budget, Choice, Float, Int, Space

logger.remove()
logger.add(sys.stderr, level="WARNING")
datasets.disable_progress_bars()
hub_quiet()
optuna.logging.set_verbosity(optuna.logging.WARNING)

EPOCHS = HARNESS.epochs
REPEATS = HARNESS.repeats
N_TRIALS = param(24, test=2)
WIDTHS = param([8, 16, 32, 64], test=[8, 16])  # the parameter axis of E2
FRACTIONS = param([0.05, 0.2], test=[0.2])  # the data axis of E3, besides 1.0
OBJECTIVE = "macro_recall"
EXAMPLE = torch.zeros(2, 187, 1)  # two beats are enough to trace the shapes

# Part 1 is the same for everyone, so its idea is the course's: "lesson 3 as handed out".
HANDED_OUT = Idea.handed_out(lesson=3)
RUNS = StudyLog.runs(
    name="lesson3-notebook-runs", idea=HANDED_OUT, objective=OBJECTIVE, choices=CHOICES
)

# the trainer's settings of every run in Part 1
TRAINER = {
    "dataset": "mitbih",
    "epochs": EPOCHS,
    "lr": 1e-3,
    "class_weight": 0.0,
    "train_fraction": 1.0,
}
# the handed-out design: every lesson 2 option off, the route stage on
ROUTE = {**vars(DESIGN.config()), **TRAINER, "route": True, "matched": False}
MATCHED = {**ROUTE, "matched": True}
ROLE = {"route": "hypothesis", "matched": "ablation"}


def n_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


def spread(config: dict, name: str) -> tuple[np.ndarray, int]:
    # `REPEATS` runs of the design `config` chooses, each from other random weights,
    # logged to RUNS under `name`; the objective per repeat, and the model's n_params
    data = HARNESS.data(config)
    make = lambda: DESIGN.build(config, n_classes=len(data.classes))
    runs = repeat(make, lambda model: HARNESS.fit(model, data, config), REPEATS)
    RUNS.log(
        name=name,
        role=ROLE[arm(config)],
        results=runs,
        dataset="mitbih",
        params=HARNESS.params(config, data),
        architecture=Architecture.of(make(), EXAMPLE),
    )
    return runs.values(OBJECTIVE), n_params(make())


def objective(space: Space, log: StudyLog):
    # what an Optuna study calls once per trial: draw a config, build the design, train
    # it, report the macro recall after every epoch, and return the last one
    def run_trial(trial: optuna.Trial) -> float:
        config = space.draw(trial)
        data = HARNESS.data(config)
        model = DESIGN.build(config, n_classes=len(data.classes))
        seen = None

        def to_optuna(epoch, fit):
            nonlocal seen
            seen = fit
            trial.report(fit.scores().get(OBJECTIVE), epoch)

        try:
            seen = HARNESS.fit(model, data, config, to_optuna)
            return seen.scores().get(OBJECTIVE)
        finally:
            log.record(
                trial,
                RunRecord(
                    role="trial",
                    dataset="mitbih",
                    config=config,
                    fixed=space.fixed(),
                    train_seconds=seen.seconds if seen else 0.0,
                    curves=seen.curves if seen else None,
                    scores=seen.scores() if seen and seen.confusion else None,
                    confusion=seen.confusion if seen else None,
                    extra=dict(data.dims),
                ),
                Architecture.of(model, EXAMPLE),
            )

    return run_trial


def search(space: Space, name: str, n_trials: int = N_TRIALS) -> StudyLog:
    # `n_trials` random trials of `space` as the Optuna study `name`
    study = StudyLog.search(
        name=name,
        space=space,
        idea=HANDED_OUT,
        objective=OBJECTIVE,
        attrs={"origin": "notebook"},
    )
    start = time.time()
    study.optimize(objective(space, study), n_trials=n_trials)
    done = study.trials(optuna.trial.TrialState.COMPLETE)
    print(
        f"{n_trials} trials in {time.time() - start:.0f}s; {len(done)} finished in all"
    )
    return study


def plot_curve(x: dict, runs: dict, xlabel: str, title: str) -> None:
    # one line per arm: the mean over the repeats with its spread, every repeat a dot
    _, ax = plt.subplots(figsize=(8, 3.6))
    for name, color in [("route", "tab:orange"), ("matched", "tab:blue")]:
        xs = np.array([x[name, point] for point in runs[name]])
        values = list(runs[name].values())
        means = np.array([v.mean() for v in values])
        stds = np.array([v.std() for v in values])
        ax.errorbar(xs, means, yerr=stds, fmt="o-", color=color, capsize=4, label=name)
        for xv, v in zip(xs, values, strict=True):
            ax.scatter([xv] * len(v), v, color=color, alpha=0.3, s=14)
    ax.set_xscale("log")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(f"{OBJECTIVE} on the validation patients")
    ax.set_title(title, loc="left", fontsize=10)
    ax.legend(title="arm")
    ax.spines[["top", "right"]].set_visible(False)
    plt.show()


def against(runs: dict, label: str) -> pd.DataFrame:
    # per point of a curve: both arms' mean and spread, and route against matched in
    # points and in sigmas (the difference of the means in standard errors)
    rows = {}
    for point in runs["route"]:
        c = Comparison.of("route", runs["route"][point], versus=runs["matched"][point])
        rows[point] = {
            "route": c.mean,
            "route std": c.std,
            "matched": runs["matched"][point].mean(),
            "matched std": runs["matched"][point].std(),
            "route - matched (pt)": c.points,
            "in sigma": c.sigmas,
        }
    table = pd.DataFrame(rows).T
    table.index.name = label
    return table


train, valid = splits(TRAIN_ROWS)
counts = pd.DataFrame(
    {"train": train.counts(), "validation": valid.counts()}, index=CLASSES
).T
display(counts.assign(beats=counts.sum(axis=1)))

The S column is the one to keep in mind: S is about one beat in seventy in train. A run
on a fraction of the beats trains on that fraction of the S beats too.

# Part 1: the notebook as handed out

## E1: the baseline, first

The floors of notebook 2.4: always N, and a linear model on the 187 steps, repeated so
you know its spread. The linear model is the baseline every later number is read
against, here and in your own study.

In [ ]:
always_n = Confusion.of(valid.labels, torch.zeros_like(valid.labels), CLASSES)
linear_config = {**LINEAR, **TRAINER}
linear_data = HARNESS.data(linear_config)
make_linear = lambda: nn.Sequential(nn.Flatten(), nn.Linear(187, 3))
linear_runs = repeat(
    make_linear, lambda model: HARNESS.fit(model, linear_data, linear_config), REPEATS
)
baseline = linear_runs.values(OBJECTIVE)
RUNS.log(
    name="baseline",
    role="baseline",
    results=linear_runs,
    dataset="mitbih",
    params=HARNESS.params(linear_config, linear_data),
    architecture=Architecture.of(make_linear(), EXAMPLE),
)
print(f"always N          macro recall {always_n.scores().macro_recall:.3f}")
print(f"linear, {REPEATS} runs    {Comparison.of('linear', baseline)}")

## E2: the route stage, its control, and a curve over parameters

The design of [`exercises/designs.py`](exercises/designs.py) is lesson 2's `Beats`: a
kernel of 5, three blocks of `width` channels, each followed by a pool of 2, and a
linear head. Lesson 3 adds one stage after the last block, on the shortened beat of 23
positions, and its control:

<svg width="640" height="150" viewBox="0 0 640 150" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="12">
  <text x="10" y="20" font-weight="bold">route</text>
  <rect x="10" y="30" width="120" height="34" fill="#eee" stroke="#555"/>
  <text x="20" y="52">blocks, pooled</text>
  <text x="140" y="44">(B, 23, w)</text>
  <line x1="130" y1="47" x2="230" y2="47" stroke="#555" marker-end="url(#a)"/>
  <rect x="230" y="26" width="200" height="42" fill="#fde5cc" stroke="#c60"/>
  <text x="240" y="44">router picks or weighs</text>
  <text x="240" y="60">n_experts experts, w -&gt; w</text>
  <line x1="430" y1="47" x2="520" y2="47" stroke="#555" marker-end="url(#a)"/>
  <text x="440" y="44">(B, 23, w)</text>
  <rect x="520" y="30" width="80" height="34" fill="#eee" stroke="#555"/>
  <text x="535" y="52">head</text>
  <text x="10" y="100" font-weight="bold">matched</text>
  <rect x="10" y="110" width="120" height="34" fill="#eee" stroke="#555"/>
  <text x="20" y="132">blocks, pooled</text>
  <text x="140" y="124">(B, 23, w)</text>
  <line x1="130" y1="127" x2="230" y2="127" stroke="#555" marker-end="url(#a)"/>
  <rect x="230" y="106" width="200" height="42" fill="#d6e6f5" stroke="#36c"/>
  <text x="240" y="124">Linear w -&gt; h, GELU, Linear h -&gt; w</text>
  <text x="240" y="140">h chosen for the same n_params</text>
  <line x1="430" y1="127" x2="520" y2="127" stroke="#555" marker-end="url(#a)"/>
  <text x="440" y="124">(B, 23, w)</text>
  <rect x="520" y="110" width="80" height="34" fill="#eee" stroke="#555"/>
  <text x="535" y="132">head</text>
  <defs><marker id="a" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto"><path d="M0,0 L8,4 L0,8 z" fill="#555"/></marker></defs>
</svg>

- **`route`** is notebook 3.1's mixture of experts, on every one of the 23 positions:
  `n_experts` experts (4 here), each a linear layer of `w -> w` with a GELU, and a router
  that reads the position and blends the experts' outputs with a softmax (`MoE`, dense).
  The shape stays `(B, 23, w)`.
- **`matched`** is its capacity-matched control (goal 3.1): the route stage replaced by
  one plain path, `w -> h -> w`, with `h` chosen so the stage has the same number of
  parameters. No router, no experts; the same path at every position.

The two arms differ in what they do with the same parameters: choose between several
narrow paths per position, or spend them on one wider path. Read it in atlas before you
run it:

```bash
atlas-code-nightly view notebooks/3_scaling_and_search/exercises
```

The cell below builds both arms at every width of the curve and counts their
parameters. **Check that they match** before anything is trained: a curve whose arms
differ in size answers another question.

In [ ]:
sizes = pd.DataFrame(
    {
        w: {
            "route": n_params(DESIGN.build({**ROUTE, "width": w}, n_classes=3)),
            "matched": n_params(DESIGN.build({**MATCHED, "width": w}, n_classes=3)),
            "h of matched": matched_hidden(w, ROUTE["n_experts"]),
        }
        for w in WIDTHS
    }
).T
sizes["difference"] = (sizes["matched"] - sizes["route"]) / sizes["route"]
sizes.index.name = "width"
display(sizes.style.format({"difference": "{:+.2%}"}))

The curve: both arms at every width, on every training beat, `REPEATS` runs each. The
width grows by doubling, so the parameters grow about fourfold per step.

**Before you run the next cell**, write down two things: at which width you expect the
curve to flatten, and whether you expect the route arm above its control by more than the
spread between runs. Notebook 3.1 found no gain for routing on electricity demand; a beat
has three classes that look different, which is one reason a router could pay here.

In [ ]:
by_width = {"route": {}, "matched": {}}
x_params = {}
for w in WIDTHS:
    for name, config in [("route", ROUTE), ("matched", MATCHED)]:
        values, size = spread({**config, "width": w}, f"{name}, width {w}")
        by_width[name][w] = values
        x_params[name, w] = size
plot_curve(
    x_params,
    by_width,
    "parameters of the model (n_params)",
    f"the curve over parameters, on every training beat, {REPEATS} runs per point",
)
display(
    against(by_width, "width")
    .style.format("{:.3f}", subset=["route", "route std", "matched", "matched std"])
    .format("{:+.1f}", subset=["route - matched (pt)", "in sigma"])
)

Read the curve and the table:

1. **Where does the curve flatten?** Look at the step from one width to the next: is the
   gain in macro recall larger than the spread of the points? A step that is within the
   spread is where more parameters stopped paying, on this data and this budget.
2. **Did routing beat its control?** The column *in sigma* is the difference of the two
   arms in standard errors. Is it more than about two at any width, and does it keep its
   sign from one width to the next? If not, the route stage bought what its parameters
   bought, and nothing more.
3. Compare every point with the linear baseline of E1. How much of the curve lies above
   it, in its own spread?

## E3: a curve over data

The same two arms at the middle width, 32, trained on a share of the beats. The points at
1.0 are E2's: the same config, so they are reused, not run again.

**Before you run the next cell**, write down what you expect: does the curve over data
flatten earlier or later than the one over parameters, and does the router need more
data than its control to pay, or less?

In [ ]:
MIDDLE = 32 if 32 in WIDTHS else WIDTHS[-1]
by_data = {"route": {}, "matched": {}}
x_data = {}
for share in [*FRACTIONS, 1.0]:
    n_train = len(fraction(TRAIN_ROWS, share).labels)
    for name, config in [("route", ROUTE), ("matched", MATCHED)]:
        if share == 1.0:
            by_data[name][share] = by_width[name][MIDDLE]
        else:
            config = {**config, "width": MIDDLE, "train_fraction": share}
            by_data[name][share], _ = spread(config, f"{name}, train_fraction {share}")
        x_data[name, share] = n_train
plot_curve(
    x_data,
    by_data,
    "training beats (n_train)",
    f"the curve over data, at width {MIDDLE}, {REPEATS} runs per point",
)
display(
    against(by_data, "train_fraction")
    .style.format("{:.3f}", subset=["route", "route std", "matched", "matched std"])
    .format("{:+.1f}", subset=["route - matched (pt)", "in sigma"])
)

1. **On which axis is this problem bound?** Put the two curves side by side: which one
   still climbs at its right end, and which has flattened? The axis that still pays is
   where your study should spend its budget.
2. At the smallest fraction a run trains on a few dozen S beats. Read the recall of S
   of those runs in `make optuna-dashboard` (the study `lesson3-notebook-runs`, the
   user attr `metrics`): is the drop on the left of the curve a drop in S?
3. Did routing beat its control anywhere on this curve, by more than about two sigma?

## E4: the handed-out search, over every capacity axis

The search below is handed out to the whole class. It draws the two parameter axes, the
data axis and the compute axis at once, and the arm:

| axis | values | what it turns |
|---|---|---|
| `width` | 8 to 64 | parameters: channels per block |
| `depth` | 1 to 4 | parameters: blocks |
| `train_fraction` | 0.05 to 1.0 | data |
| `epochs` | a quarter of `EPOCHS` to `EPOCHS` | compute |
| `route` | off, on | the route stage |
| `matched` | off, on | with `route` on, its control in its place |

`route` off is the design without either stage (the arm `plain`), so a trial falls in one
of three arms: `plain` half of the time, `route` and `matched` a quarter each. As in
lessons 1 and 2, the search samples at random, is not seeded and prunes nothing: a pruner
stops a trial on its first epochs, and a curve over compute needs every trial to run the
epochs it drew. The class's trials pooled are one random search of a few hundred trials,
enough to read curves per arm that 24 trials cannot.

In [ ]:
HANDED_OUT_SPACE = Space(
    {
        "dataset": "mitbih",
        "width": Int(8, 64, log=True),
        "depth": Int(1, 4),
        "train_fraction": Float(0.05, 1.0, log=True),
        "epochs": Int(max(1, EPOCHS // 4), EPOCHS),
        "route": Choice([False, True]),
        "matched": Choice([False, True]),
        "n_experts": 4,
        "routing": "dense",
        "parallel": False,
        "fold": False,
        "skip": False,
        "excite": False,
        "state": False,
        "gate": False,
        "class_weight": 0.0,
        "lr": 1e-3,
    }
)
handed_out = search(HANDED_OUT_SPACE, "lesson3-notebook")
table = trial_rows(handed_out)

axes = ["n_params", "n_train", "epochs"]
fig, panels = plt.subplots(1, len(axes), figsize=(15, 3.4), sharey=True)
colors = {"plain": "tab:gray", "route": "tab:orange", "matched": "tab:blue"}
for ax, axis in zip(panels, axes, strict=True):
    for name, group in table.groupby("arm"):
        ax.scatter(
            group[axis], group["value"], color=colors[name], alpha=0.7, label=name
        )
    ax.set_xlabel(axis)
    if axis != "epochs":
        ax.set_xscale("log")
panels[0].set_ylabel(OBJECTIVE)
panels[-1].legend(title="arm")
plt.suptitle("every trial of the search, along each capacity axis", x=0.1, ha="left")
plt.tight_layout()
plt.show()
for axis in ["n_params", "train_fraction"]:
    display(curve(table, axis).round(3))

The tables put the trials in bins along an axis and give per bin and arm the mean, its
standard error (`se`) and the number of trials (`n`). A bin with one or two trials of an
arm says almost nothing; that is what 24 trials spread over three arms and four axes
buy. Which panel shows a pattern already, and which would you want the class's trials
for?

Your runs of Part 1 are now in `lesson3-notebook` and `lesson3-notebook-runs`.

# Part 2: your own study

The loop is the one of lessons 1 and 2, on lesson 3's design; `docs/your-study.md` walks
its nine steps. You work in [`exercises/`](exercises/):

| file | what it is | what you do with it |
|---|---|---|
| [`exercises/designs.py`](exercises/designs.py) | `Beats`, one typed method per stage; lesson 2's blocks and `route` and `matched` are options | read it with atlas; add a stage when your idea needs another block, and a control of the same parameter count for it |
| [`exercises/study.py`](exercises/study.py) | six settings at the top (`STUDY`, `IDEA`, `EXPECTS`, `OBJECTIVE`, `SPACE`, `MINUTES`) and the code that runs the study | fill in the settings, then `make study LESSON=3` |
| [`exercises/harness.py`](exercises/harness.py) | the heartbeats split by patient, `train_fraction`, the trainer, the linear baseline, the curves and the control | nothing: it is given |

What the study does, in order: it measures the linear baseline, searches your `SPACE`,
reruns the winner and compares it with the baseline in sigmas (as in lesson 2), then
prints the capacity curves of its trials per arm along every capacity axis it searched,
and last runs the winner's config with the route stage and with its control, each
`REPEATS` times, and prints the two against each other in sigmas.

**Your study searches at least two capacity axes** (parameters, data, compute) and the
route stage with its control. Which two, and over which ranges, is yours to decide, and
so is the question the curve answers.

## The open questions

Every question below is one bet a study can test; the class spreads over them, so that
the next lesson opens on a map of what capacity bought, in sigmas, not a leaderboard. A
bet that did not pay is a result on that map.

- **Does routing pay with more data, or with less?** A router has to learn which expert
  takes which position. Does the gap between `route` and `matched` change sign along
  `train_fraction`?
- **Do more experts buy anything?** `n_experts` sets how many paths the router chooses
  between, and the control grows with it. At a fixed width, does a curve over
  `n_experts` flatten, and where?
- **Sparse against dense.** `routing="sparse"` runs one expert per position (notebook
  3.1's `SparseMoE`): it trains as many parameters and uses fewer per position (goal
  3.4). Does it lose against the dense mixture at the same `n_params`, and does its
  router collapse onto one expert?
- **Does S need parameters, data or compute?** S is rare. Read the recall of S of your
  trials along each capacity axis: which one moves it?
- **Is the problem bound on compute?** `epochs` is an axis too. Is a small model trained
  longer worth as much as a large one trained briefly, at the same seconds?
- **Does a lesson 2 block change the curve?** `skip`, `state`, `gate` and the others are
  still options. Does one of them shift the curve over parameters, against a control of
  the same size?

A question of your own is welcome, as long as it is a bet you can state before you run.

## The steps

Run every command in a terminal, from the root of the repository.

1. **Design.** Read the design in atlas:

   ```bash
   atlas-code-nightly view notebooks/3_scaling_and_search/exercises
   ```

2. **Build.** Change the design if your question needs a block it does not have, then
   check that every combination builds and that the route stage and its control have the
   same parameters:

   ```bash
   uv run notebooks/3_scaling_and_search/exercises/designs.py
   ```

3. **Baseline.** Nothing to do: `make study LESSON=3` measures the linear baseline before
   it searches.
4. **Budget and space.** `STUDY` is your first name and what it studies:
   `"sam-experts-on-little-data"`. `IDEA` says what you investigate in your own words;
   your first idea builds on Part 1 (`builds_on=Idea.handed_out(3).name`), a later one on
   an idea of the class, by its name. `EXPECTS` is what you expect, and why, written
   before you run. `SPACE` searches at least two capacity axes and the arm, and fixes the
   rest; every option of the design is an axis, searched or fixed. `MINUTES` is your
   budget; the cell below shows what it buys.
5. **Search:** `make study LESSON=3`. It searches, reruns the winner, prints the curves
   per arm and the route stage against its control in sigmas.
6. **Read** it in `make optuna-dashboard`: per axis, what it did to the macro recall,
   the `n_params` of every trial, and the recall of S of your best trials.
7. **Explain:** `make page LESSON=3` builds your page of the lesson from your store; its
   step on the page says what to fill in.
8. **Share:** `make share LESSON=3`.
9. **Build on:** see the class in `make dashboard LESSON=3`. Your next idea builds on one
   of the class.

In [ ]:
MINUTES = 15  # the time you give your study on this machine
AXES = 4  # how many axes your SPACE searches
PART1 = StudyLog.load(name="lesson3-notebook")
SECONDS_PER_TRIAL = PART1.seconds_per_trial(epochs=EPOCHS)


def trials_in(minutes: float) -> int:
    # the trials that fit in `minutes`, once the baseline and the winner's rerun
    # (REPEATS runs each) are set aside
    return Budget.at(
        minutes=minutes,
        seconds_per_trial=SECONDS_PER_TRIAL,
        measured_on=PART1.name,
        epochs=EPOCHS,
        repeats=REPEATS,
    ).n_trials


minutes = np.arange(1, 61)
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(minutes, [trials_in(m) for m in minutes], color="tab:blue")
ax.scatter([MINUTES], [trials_in(MINUTES)], color="tab:orange", s=80, zorder=3)
ax.annotate(
    f"{MINUTES} minutes: {trials_in(MINUTES)} trials, "
    f"{trials_in(MINUTES) / AXES:.0f} per searched axis",
    (MINUTES, trials_in(MINUTES)),
    xytext=(10, -14),
    textcoords="offset points",
)
ax.set_xlabel("minutes you give the study")
ax.set_ylabel("trials that fit")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title(
    f"one trial costs {SECONDS_PER_TRIAL:.1f}s here, at {EPOCHS} epochs "
    "(measured by the search of Part 1)",
    loc="left",
    fontsize=10,
)
plt.show()

The study also runs the route stage and its control at the winner, `REPEATS` runs each,
after the trials this plot counts: give it a few minutes more than the trials alone
would need. A trial on a small `train_fraction` or few `epochs` costs less than the
trial this plot assumes, so a search over those axes fits more trials than it shows.

## Hand in your runs before the next lesson

```bash
make share LESSON=3
```

It hands in Part 1 (`lesson3-notebook`, `lesson3-notebook-runs`) and every study of
yours (`lesson3-study-*`), each run with the design script it ran. Name the ideas of
classmates you built on in `builds_on`; the class page shows who built on whom.